
# E1 Modules A/C/D: operating characteristics, tuning, and variance ratio

This is the combined Phase-3 runner specified in `REVISION_PLAN.md` P3-T2.
It executes Module A (840 cells), Module C (243 cells), and Module D
(216 cells), for 1,299 cells total. Each cell uses 5,000 outer repetitions,
is checkpointed to Google Drive, and is safe to resume after a Colab
disconnect. The empirical family requires the real M x 2 loss matrix used
by the E1 row-bootstrap; the setup cell finds it on Drive or offers the
standard Colab upload dialog.


In [ ]:

import itertools
import os
import pathlib
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_URL = "https://github.com/hugogobato/Test-Informed-Simulation-Count-Algorithm-TISCA.git"
LOCAL_REPO = "/content/Test-Informed-Simulation-Count-Algorithm-TISCA"
CLONED_REPO = "/content/TISCA_repo"

if os.path.isdir(os.path.join(LOCAL_REPO, "tisca", "python")):
    SOURCE_ROOT = os.path.join(LOCAL_REPO, "tisca", "python")
elif os.path.isdir(os.path.join(CLONED_REPO, "tisca", "python")):
    SOURCE_ROOT = os.path.join(CLONED_REPO, "tisca", "python")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONED_REPO], check=True)
    SOURCE_ROOT = os.path.join(CLONED_REPO, "tisca", "python")

assert os.path.isdir(SOURCE_ROOT), f"TISCA Python package not found at {SOURCE_ROOT}"
sys.path.insert(0, SOURCE_ROOT)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("(Not on Colab / Drive mount skipped):", exc)

if os.path.isdir("/content/drive/MyDrive"):
    DRIVE_ROOT = "/content/drive/MyDrive/TISCA_E1"
else:
    DRIVE_ROOT = "/content/TISCA_E1"
    print("[WARN] Drive is not mounted; checkpointing to", DRIVE_ROOT)
os.makedirs(DRIVE_ROOT, exist_ok=True)

from tisca.outermc import engine, summarize_ocs
from tisca import multiplicity

ALPHA = 0.05
DELTA = 0.5
JMAX = 1000
MATRIX_CANDIDATES = [
    os.path.join(DRIVE_ROOT, "E1_empirical_loss_matrix.npy"),
    os.path.join(DRIVE_ROOT, "E1_empirical_loss_matrix.csv"),
    "/content/E1_empirical_loss_matrix.npy",
    "/content/E1_empirical_loss_matrix.csv",
]


def load_empirical_matrix():
    """Find or upload the real M x 2 loss matrix used by family (g)."""
    path = next((p for p in MATRIX_CANDIDATES if os.path.exists(p)), None)
    if path is None:
        try:
            from google.colab import files
            print("Upload E1_empirical_loss_matrix.npy or .csv (M x 2) when prompted.")
            uploaded = files.upload()
            if uploaded:
                name = next(iter(uploaded))
                path = os.path.join("/content", name)
        except Exception as exc:
            print("(Upload skipped):", exc)
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(
            "The empirical family needs the real M x 2 loss matrix. "
            "Place E1_empirical_loss_matrix.npy/.csv in the Drive folder or upload it."
        )
    if path.lower().endswith(".npy"):
        matrix = np.load(path)
    else:
        matrix = pd.read_csv(path, header=None).to_numpy(dtype=float)
    matrix = np.asarray(matrix, dtype=float)
    if matrix.ndim != 2 or matrix.shape[1] != 2 or matrix.shape[0] < 2:
        raise ValueError(f"empirical matrix must have shape (M, 2), got {matrix.shape}")
    if not np.all(np.isfinite(matrix)):
        raise ValueError("empirical matrix contains non-finite values")
    print("[PASS] empirical matrix:", matrix.shape, "from", path)
    return matrix


EMPIRICAL_MATRIX = load_empirical_matrix()


def _append(row, path):
    frame = pd.DataFrame([row])
    header = not os.path.exists(path) or os.path.getsize(path) == 0
    frame.to_csv(path, mode="a", header=header, index=False)


def run_grid(grid, output_name):
    """Run and checkpoint a deterministic grid, resuming completed cell IDs."""
    output_file = os.path.join(DRIVE_ROOT, output_name)
    error_file = output_file.replace("_results.csv", "_errors.csv")
    done = set()
    if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
        old = pd.read_csv(output_file)
        if "cell_id" not in old.columns:
            raise ValueError(f"existing checkpoint lacks cell_id: {output_file}")
        done = set(old["cell_id"].astype(str))

    pending = [c for c in grid if c["cell_id"] not in done]
    print(f"{output_name}: {len(done)} completed, {len(pending)} pending, {len(grid)} expected")
    failures = []
    started = time.time()
    for cell in tqdm(pending, desc=output_name, unit="cell"):
        try:
            cfg = dict(cell["config"])
            summary, _, _ = engine.run_e1(cfg)
            row = summarize_ocs([summary]).iloc[0].to_dict()
            row.update(cell["factors"])
            row.update(
                cell_id=cell["cell_id"],
                module=cell["module"],
                projected_R=cell["config"]["R"],
                bootstrap_B=cell["config"].get("B", np.nan),
            )
            _append(row, output_file)
        except Exception as exc:
            print("[FAIL]", cell["cell_id"], repr(exc))
            failures.append({"cell_id": cell["cell_id"], "error": repr(exc)})

    if failures:
        pd.DataFrame(failures).to_csv(error_file, index=False)
        raise RuntimeError(f"{len(failures)} cells failed; see {error_file}")

    result = pd.read_csv(output_file)
    if set(result["cell_id"].astype(str)) != {c["cell_id"] for c in grid}:
        missing = sorted({c["cell_id"] for c in grid} - set(result["cell_id"].astype(str)))
        raise RuntimeError(f"checkpoint incomplete; missing {len(missing)} cells, first={missing[:3]}")
    if result["cell_id"].duplicated().any():
        raise RuntimeError("duplicate cell_id detected in checkpoint")
    print(f"[PASS] {len(result)} rows in {output_file}; elapsed {time.time() - started:.0f}s")
    return output_file, result


def download_fallback(output_file):
    try:
        from google.colab import files
        files.download(output_file)
        print("Downloaded:", output_file)
    except Exception as e:
        print("(Not on Colab / download skipped):", e)



## Define the three grids

Module A spans all seven families, five correlations, four effect levels,
and six designs. Module C varies pilot size, checkpoint batch, family,
correlation, and the three adaptive/two-stage designs at the planning
alternative. Module D varies the marginal variance ratio, family,
correlation, effect, and design.


In [ ]:

def make_acd_grid():
    cells = []
    index = 0

    # Module A: F(7) x rho(5) x theta(4) x Design(6), K=1, J0=50, B=50.
    for family, rho, theta_mult, design in itertools.product(
        ["normal", "lognormal", "gamma", "mix", "beta", "t3", "empirical"],
        [-0.3, 0.0, 0.3, 0.6, 0.9],
        [0.0, 0.5, 1.0, 2.0],
        ["D1", "D2", "D3", "D4", "D5", "D6"],
    ):
        factors = dict(module_cell="A", family=family, rho=rho,
                       theta_mult=theta_mult, design=design, J0=50, B=50,
                       sigma_ratio=1.0)
        config = {
            "design": design, "family": family, "rho": rho,
            "sigma_a": 1.0, "sigma_b": 1.0, "theta": DELTA * theta_mult,
            "sigma_D": None, "R": 5000, "J0": 50, "Jmax": JMAX,
            "alpha": ALPHA, "alpha_adj": ALPHA, "mode": 1, "delta": DELTA,
            "power_target": 0.80, "gamma": 0.20, "correction": "none",
            "K": 1, "matrix": EMPIRICAL_MATRIX if family == "empirical" else None,
            "seed": 100000 + index, "fixed_J": None, "mcse": 0.05,
            "batch": 50, "B": 50,
        }
        cells.append(dict(cell_id=f"A_{index:04d}", module="A",
                          factors=factors, config=config))
        index += 1

    # Module C: J0(3) x B(3) x F(3) x rho(3) x Design(3), theta=delta.
    for J0, B, family, rho, design in itertools.product(
        [25, 50, 100], [25, 50, 100], ["normal", "lognormal", "gamma"],
        [-0.3, 0.3, 0.9], ["D2", "D3", "D4"],
    ):
        factors = dict(module_cell="C", family=family, rho=rho,
                       theta_mult=1.0, design=design, J0=J0, B=B,
                       sigma_ratio=1.0)
        config = {
            "design": design, "family": family, "rho": rho,
            "sigma_a": 1.0, "sigma_b": 1.0, "theta": DELTA,
            "sigma_D": None, "R": 5000, "J0": J0, "Jmax": JMAX,
            "alpha": ALPHA, "alpha_adj": ALPHA, "mode": 1, "delta": DELTA,
            "power_target": 0.80, "gamma": 0.20, "correction": "none",
            "K": 1, "matrix": None, "seed": 200000 + index,
            "fixed_J": None, "mcse": 0.05, "batch": B, "B": B,
        }
        cells.append(dict(cell_id=f"C_{index - 840:04d}", module="C",
                          factors=factors, config=config))
        index += 1

    # Module D: sigma-ratio(2) x F(3) x rho(3) x theta(2) x Design(6).
    for sigma_ratio, family, rho, theta_mult, design in itertools.product(
        [1.0, 2.0], ["normal", "lognormal", "gamma"], [-0.3, 0.3, 0.9],
        [0.0, 1.0], ["D1", "D2", "D3", "D4", "D5", "D6"],
    ):
        factors = dict(module_cell="D", family=family, rho=rho,
                       theta_mult=theta_mult, design=design, J0=50, B=50,
                       sigma_ratio=sigma_ratio)
        config = {
            "design": design, "family": family, "rho": rho,
            "sigma_a": sigma_ratio, "sigma_b": 1.0,
            "theta": DELTA * theta_mult, "sigma_D": None, "R": 5000,
            "J0": 50, "Jmax": JMAX, "alpha": ALPHA, "alpha_adj": ALPHA,
            "mode": 1, "delta": DELTA, "power_target": 0.80, "gamma": 0.20,
            "correction": "none", "K": 1, "matrix": None,
            "seed": 300000 + index, "fixed_J": None, "mcse": 0.05,
            "batch": 50, "B": 50,
        }
        cells.append(dict(cell_id=f"D_{index - 1083:04d}", module="D",
                          factors=factors, config=config))
        index += 1

    assert len(cells) == 1299, len(cells)
    assert sum(c["module"] == "A" for c in cells) == 840
    assert sum(c["module"] == "C" for c in cells) == 243
    assert sum(c["module"] == "D" for c in cells) == 216
    return cells


ACD_GRID = make_acd_grid()
print("Module A/C/D cells:", {m: sum(c["module"] == m for c in ACD_GRID) for m in "ACD"})


In [ ]:

OUTPUT_FILE, RESULTS = run_grid(ACD_GRID, "E1_modA_C_D_results.csv")
print(RESULTS.groupby("module").size())
print(RESULTS.head())



## Basic completion checks

The checkpoint must contain exactly one row per declared cell. The D6
normal cells in Module A provide the oracle sanity check. Module B is
deliberately separate because its bootstrap-heavy correction grid is
split into two notebooks.


In [ ]:

assert len(RESULTS) == 1299
assert RESULTS["cell_id"].is_unique
d6 = RESULTS[(RESULTS["module"] == "A") & (RESULTS["design"] == "D6") &
             (RESULTS["family"] == "normal")]
assert len(d6) == 20, len(d6)
print("[PASS] A/C/D complete; D6 normal oracle rows:", len(d6))
download_fallback(OUTPUT_FILE)
